
# Assignment 4 for Course 1MS041
Make sure you pass the `# ... Test` cells and
 submit your solution notebook in the corresponding assignment on the course website. You can submit multiple times before the deadline and your highest score will be used.

---
## Assignment 4, PROBLEM 1
Maximum Points = 24


    This time the assignment only consists of one problem, but we will do a more comprehensive analysis instead.

Consider the dataset `Corona_NLP_train.csv` that you can get from the course website [git](https://github.com/datascience-intro/1MS041-2024/blob/main/notebooks/data/Corona_NLP_train.csv). The data is "Coronavirus tweets NLP - Text Classification" that can be found on [kaggle](https://www.kaggle.com/datasets/datatattle/covid-19-nlp-text-classification). The data has several columns, but we will only be working with `OriginalTweet`and `Sentiment`.

1. [3p] Load the data and filter out those tweets that have `Sentiment`=`Neutral`. Let $X$ represent the `OriginalTweet` and let 
    $$
        Y = 
        \begin{cases}
        1 & \text{if sentiment is towards positive}
        \\
        0 & \text{if sentiment is towards negative}.
        \end{cases}
    $$
    Put the resulting arrays into the variables $X$ and $Y$. Split the data into three parts, train/test/validation where train is 60% of the data, test is 15% and validation is 25% of the data. Do not do this randomly, this is to make sure that we all did the same splits (we are in this case assuming the data is IID as presented in the dataset). That is [train,test,validation] is the splitting layout.

2. [4p] There are many ways to solve this classification problem. The first main issue to resolve is to convert the $X$ variable to something that you can feed into a machine learning model. For instance, you can first use [`CountVectorizer`](https://scikit-learn.org/1.5/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) as the first step. The step that comes after should be a `LogisticRegression` model, but for this to work you need to put together the `CountVectorizer` and the `LogisticRegression` model into a [`Pipeline`](https://scikit-learn.org/1.5/modules/generated/sklearn.pipeline.Pipeline.html#sklearn.pipeline.Pipeline). Fill in the variable `model` such that it accepts the raw text as input and outputs a number $0$ or $1$, make sure that `model.predict_proba` works for this. **Hint: You might need to play with the parameters of LogisticRegression to get convergence, make sure that it doesn't take too long or the autograder might kill your code**
3. [3p] Use your trained model and calculate the precision and recall on both classes. Fill in the corresponding variables with the answer.
4. [3p] Let us now define a cost function
    * A positive tweet that is classified as negative will have a cost of 1
    * A negative tweet that is classified as positive will have a cost of 5
    * Correct classifications cost 0
    
    complete filling the function `cost` to compute the cost of a prediction model under a certain prediction threshold (recall our precision recall lecture and the `predict_proba` function from trained models). 

5. [4p] Now, we wish to select the threshold of our classifier that minimizes the cost, fill in the selected threshold value in value `optimal_threshold`.
6. [4p] With your newly computed threshold value, compute the cost of putting this model in production by computing the cost using the validation data. Also provide a confidence interval of the cost using Hoeffdings inequality with a 99% confidence.
7. [3p] Let $t$ be the threshold you found and $f$ the model you fitted (one of the outputs of `predict_proba`), if we define the random variable
    $$
        C = (1-1_{f(X)\geq t})Y+5(1-Y)1_{f(X) \geq t}
    $$
    then $C$ denotes the cost of a randomly chosen tweet. In the previous step we estimated $\mathbb{E}[C]$ using the empirical mean. However, since the threshold is chosen to minimize cost it is likely that $C=0$ or $C=1$ than $C=5$ as such it will have a low variance. Compute the empirical variance of $C$ on the validation set. What would be the confidence interval if we used Bennett's inequality instead of Hoeffding in point 6 but with the computed empirical variance as our guess for the variance?

In [ ]:

# Part 1

# Load the data from the file specified in the problem definition and make sure that it is loaded using
# the search path `data/Corona_NLP_train.csv`. This is to make sure the autograder and your computer have the same
# file path and can load the data correctly.

# Contrary to how many other problems are structured, this problem actually requires you to
# have X on the shape (n_samples, ) that is a 1-dimensional array. Otherwise it will cause a bunch
# of errors in the autograder or also in for instance CountVectorizer.

# Make sure that all your data is numpy arrays and not pandas dataframes or series.
import numpy as np
import pandas as pd
from Utils import train_test_validation

df = pd.read_csv("data/Corona_NLP_train.csv", encoding="ISO-8859-1")
df = df[df["Sentiment"] != "Neutral"].copy()
X = df["OriginalTweet"].astype(str).to_numpy()
Y = df["Sentiment"].astype(str).str.contains("Positive").astype(int).to_numpy()

X_train, X_test, X_valid, Y_train, Y_test, Y_valid = train_test_validation(
    X, Y, test_size=0.15, validation_size=0.25, random_state=0, shuffle=False
)

(33444,) (33444,)
(20066,) (5016,) (8362,)
(20066,) (5016,) (8362,)


In [ ]:

# Part 2

# Train a machine learning model or pipeline that can take the raw strings from X and predict Y=0,1 depending on the
# sentiment of the tweet. Store the trained model in the variable `model`.
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("vec", CountVectorizer(stop_words="english", max_features=50000)),
    ("clf", LogisticRegression(max_iter=1000, solver="liblinear"))
])

model.fit(X_train, Y_train)

[0 1]


In [5]:

# Part 3

# Evaluate the model on the test set and calculate precision, and recall on both classes. Store the results in the
# variables `precision_0`, `precision_1`, `recall_0`, `recall_1`.
from sklearn.metrics import precision_score, recall_score

y_pred = model.predict(X_test)
# For class 0
precision_0 = precision_score(Y_test, y_pred, pos_label=0)
recall_0    = recall_score(Y_test, y_pred, pos_label=0)

# For class 1
precision_1 = precision_score(Y_test, y_pred, pos_label=1)
recall_1    = recall_score(Y_test, y_pred, pos_label=1)

In [13]:

# Part 4
import numpy as np
def cost(model,threshold,X,Y):
    # Hint, make sure that the model has a predict_proba method
    # think about how the decision is made based on the probabilities
    # and how the threshold can be used to make the decision.
    # For reference take a look at the lecture notes "Bayes classifier"
    # which contains how the decision is made based on the probabilities when the threshold is 0.5.
    
    # Fill in what is missing to compute the cost and return it
    # Note that we are interested in average cost

    # Get predicted probabilities
    probs = model.predict_proba(X)
    
    # Find column corresponding to class 1
    if hasattr(model, "classes_"):
        class_1_index = int(np.where(model.classes_ == 1)[0][0])
        p_positive = probs[:, class_1_index]
    else:
        # Autograder dummy model: assume probs[:, 1] corresponds to class 1
        p_positive = probs[:, 1]        
    # Apply threshold to make predictions
    Y_pred = (p_positive >= threshold).astype(int)
    
    # False negatives: true = 1, predicted = 0
    fn = (Y == 1) & (Y_pred == 0)
    
    # False positives: true = 0, predicted = 1
    fp = (Y == 0) & (Y_pred == 1)
    
    # Compute costs
    costs = fn.astype(int) * 1 + fp.astype(int) * 5
    return float(np.mean(costs))

In [ ]:

# Part 5

# Find the optimal threshold for the model on the test set. Store the threshold in the variable `optimal_threshold`
# and the cost at the optimal threshold in the variable `cost_at_optimal_threshold` evaluated on the test set.
import numpy as np

# Try thresholds between 0 and 1
thresholds = np.linspace(0, 1, 101)

costs = []

for t in thresholds:
    c = cost(model, t, X_test, Y_test)
    costs.append(c)

# Find threshold with minimum cost
min_index = np.argmin(costs)

optimal_threshold = thresholds[min_index]
cost_at_optimal_threshold = costs[min_index]

Optimal threshold: 0.76
Cost at optimal threshold: 0.2982456140350877


In [9]:

# Part 6
import numpy as np
from Utils import compute_confidence_interval_bounded

# Cost on validation set using optimal threshold
cost_at_optimal_threshold_valid = cost(
    model, optimal_threshold, X_valid, Y_valid
)

# Get probabilities
probs_valid = model.predict_proba(X_valid)

# Find index for class 1
class_1_index = np.where(model.classes_ == 1)[0][0]
p_positive_valid = probs_valid[:, class_1_index]

# Apply threshold
Y_pred_valid = (p_positive_valid >= optimal_threshold).astype(int)

# Compute per-sample costs
fn = (Y_valid == 1) & (Y_pred_valid == 0)   # false negatives
fp = (Y_valid == 0) & (Y_pred_valid == 1)   # false positives

costs_valid = fn.astype(int) * 1 + fp.astype(int) * 5

# 99% confidence interval using Hoeffding
cost_interval_valid = compute_confidence_interval_bounded(
    costs_valid,
    delta=0.01,
    min_value=0,
    max_value=5
)
assert(type(cost_interval_valid) == tuple)
assert(len(cost_interval_valid) == 2)

In [12]:

# Part 7
import numpy as np
from Utils import bennett_epsilon  # if this exists in your Utils.py

# --- Recompute per-sample costs on validation (same as Part 6) ---
probs_valid = model.predict_proba(X_valid)
class_1_index = np.where(model.classes_ == 1)[0][0]
p_positive_valid = probs_valid[:, class_1_index]
Y_pred_valid = (p_positive_valid >= optimal_threshold).astype(int)

fn = (Y_valid == 1) & (Y_pred_valid == 0)
fp = (Y_valid == 0) & (Y_pred_valid == 1)

costs_valid = fn.astype(int) * 1 + fp.astype(int) * 5  # C samples in {0,1,5}

# 1) empirical variance of C on validation
variance_of_C = float(np.var(costs_valid, ddof=0))

# 2) Bennett 99% CI for E[C]
n = len(costs_valid)
mean_C = float(np.mean(costs_valid))

# C is bounded in [0,5] so b = 5
eps = bennett_epsilon(n, 5, np.sqrt(variance_of_C), 0.01)

interval_of_C = (max(0.0, mean_C - eps), min(5.0, mean_C + eps))
assert(type(interval_of_C) == tuple)
assert(len(interval_of_C) == 2)

Numerical error -9.194034422677078e-17
